In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# SQL Server to Databricks Pipeline Generation

This notebook generates optimized SQL Server to Databricks ingestion pipelines using Databricks Asset Bundles.

## Process Overview
1. **Load balancing**: Groups tables into pipeline configurations
2. **YAML generation**: Creates Databricks Asset Bundle YAML files

## Prerequisites
- Upload the `load_balancing` and `deployment` directories to your Databricks workspace
- Ensure you have a CSV file with your source table list
- Have a Databricks connection configured for SQL Server

In [0]:
%pip install pyyaml

## Configure Parameters

**IMPORTANT**: Modify these parameters for your environment before running!

In [0]:
from databricks.sdk import WorkspaceClient
from utilities import load_input_csv
from sqlserver.connector import SQLServerConnector

w = WorkspaceClient()

# Automatically get current user email
username = w.current_user.me().user_name

# Automatically get workspace host URL
workspace_host = w.config.host

# Automatically construct output directory
WORKSPACE_HOST = workspace_host
ROOT_PATH = f'/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}'



In [0]:
INPUT_CSV = 'examples/tapworks/pipeline_config.csv'
OUTPUT_DIR = f'/Workspace/Users/{username}/lakehouse-tapworks/sqlserver/examples/tapworks/deployment'

df = load_input_csv(INPUT_CSV)

# Create connector
connector = SQLServerConnector()

# Generate pipelines
result = connector.run_complete_pipeline_generation(
df=df,
output_dir=OUTPUT_DIR,
targets={
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
},
max_tables_per_gateway=250,
max_tables_per_pipeline=250
)

## Example: Using Custom Default Values and Overrides

You can use `default_values` and `override_input_config` parameters for advanced configuration:

**`default_values` Parameter:**
- Provides custom default values for optional columns
- Merged with built-in defaults (your values take precedence)
- Applied only to missing/empty values in the CSV

**`override_input_config` Parameter:**
- Forces specific column values for ALL rows
- Overwrites any existing values in the CSV
- Useful for environment-specific overrides

**Example Use Cases:**
- Override schedule for testing (e.g., disable auto-runs)
- Force all pipelines to use specific gateway settings
- Set custom defaults for columns not in your CSV

In [0]:
INPUT_CSV = 'examples/tapworks/pipeline_config.csv'
OUTPUT_DIR = f'/Workspace/Users/{username}/lakehouse-tapworks/sqlserver/examples/tapworks/deployment'

df = load_input_csv(INPUT_CSV)

# Create connector
connector = SQLServerConnector()

# Generate pipelines
result = connector.run_complete_pipeline_generation(
df=df,
output_dir=OUTPUT_DIR,
targets={
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
},
# default_values={'project_name': 'my_sql_project'},
max_tables_per_gateway=250,
max_tables_per_pipeline=250,
override_input_config={"gateway_worker_type" : "m6d.xlarge", "gateway_driver_type": "m6d.xlarge"}
)

## no worker/driver type specfied here to replace the empty ones in config -> gateway will use the platform defaults

In [0]:
INPUT_CSV = 'examples/mixed/pipeline_config.csv'
OUTPUT_DIR = f'/Workspace/Users/{username}/lakehouse-tapworks/sqlserver/examples/mixed/deployment'

df = load_input_csv(INPUT_CSV)

# Create connector
connector = SQLServerConnector()

# Generate pipelines
result = connector.run_complete_pipeline_generation(
df=df,
output_dir=OUTPUT_DIR,
targets={
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
},
max_tables_per_gateway=250,
max_tables_per_pipeline=250
)

## worker/driver type are specfied here to replace the empty ones in config -> gateway will use these specified ones

In [0]:
INPUT_CSV = 'examples/mixed/pipeline_config.csv'
OUTPUT_DIR = f'/Workspace/Users/{username}/lakehouse-tapworks/sqlserver/examples/mixed/deployment'

df = load_input_csv(INPUT_CSV)

# Create connector
connector = SQLServerConnector()

# Generate pipelines
result = connector.run_complete_pipeline_generation(
df=df,
output_dir=OUTPUT_DIR,
targets={
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
},
default_values={"gateway_worker_type" : "m7d.xlarge", "gateway_driver_type": "m7d.xlarge"},
max_tables_per_gateway=250,
max_tables_per_pipeline=250
)



In [0]:
# List generated files
print("\nGenerated DAB Project Structure:")
print(f"\n{OUTPUT_DIR}/")

# Check if files exist
import os
files_to_check = [
    'databricks.yml',
    'resources/gateways.yml',
    'resources/pipelines.yml'
]

for file in files_to_check:
    full_path = os.path.join(OUTPUT_DIR, file)
    exists = "✓" if os.path.exists(full_path) else "✗"
    print(f"  {exists} {file}")

In [0]:
# # Create widgets for interactive configuration
# dbutils.widgets.text("input_csv", "load_balancing/examples/example_config.csv", "Input CSV Path")
# dbutils.widgets.text("gateway_catalog", "jack_demos", "Gateway Catalog")
# dbutils.widgets.text("gateway_schema", "ingestion_schema", "Gateway Schema")
# dbutils.widgets.text("project_name", "my_project", "Project Name")
# dbutils.widgets.text("connection_name", "conn_1", "Connection Name")
# dbutils.widgets.text("max_tables", "1000", "Max Tables Per Group")
# dbutils.widgets.text("output_dir", "/Workspace/Users/your-email/dab_project", "Output Directory")

# # Get widget values
# INPUT_CSV = dbutils.widgets.get("input_csv")
# GATEWAY_CATALOG = dbutils.widgets.get("gateway_catalog")
# GATEWAY_SCHEMA = dbutils.widgets.get("gateway_schema")
# PROJECT_NAME = dbutils.widgets.get("project_name")
# DEFAULT_CONNECTION_NAME = dbutils.widgets.get("connection_name")
# MAX_TABLES_PER_GROUP = int(dbutils.widgets.get("max_tables"))
# OUTPUT_DIR = dbutils.widgets.get("output_dir")